# 🧬 Transcription Factor Binding Prediction with OmniGenBench

Welcome to this tutorial where we'll explore how to predict Transcription Factor Binding (TFB) sites on DNA sequences using **OmniGenBench**. This guide is designed to walk you through a complete genomic deep learning project, from understanding the fundamental concepts to deploying a trained model.

### 1. The Biological Challenge: What is Transcription Factor Binding?

**Transcription Factors (TFs)** are essential proteins that regulate gene expression, the process of turning genes 'on' or 'off'. They do this by binding to specific DNA sequences. Identifying these binding sites is a critical step in understanding gene regulation, which is fundamental to comprehending cellular function, development, and disease.

However, experimentally identifying TF binding sites across the entire genome is a slow and expensive process. This is where computational methods, particularly deep learning, can make a significant impact.

### 2. The Data: An Introduction to the DeepSEA Dataset

To train a model for this task, we need a labeled dataset. We will use the **DeepSEA dataset**, a well-known benchmark in genomics.

- **What it contains**: It consists of 1000-base-pair DNA sequences.
- **What it labels**: Each sequence is associated with 919 binary labels. These labels indicate the presence or absence of various chromatin features, including TF binding, DNase I sensitivity, and histone marks.
- **Our Goal**: We will use these sequences and labels to train a model that can predict which of the 919 features are present for any given DNA sequence.

### 3. The Tool: From Language Models to Genomic Foundation Models

#### The Rise of Language Models
In recent years, **Language Models (LMs)** like BERT have revolutionized Natural Language Processing (NLP). Trained on vast amounts of text, they learn the underlying patterns of language-grammar, context, and even semantics. This allows them to be "fine-tuned" for a wide range of specific tasks (e.g., translation, summarization, question answering).

#### A New Paradigm in Genomics: Genomic Foundation Models (GFMs)
The same principles can be applied to biology. The "language of life" is written in DNA, a sequence of A, C, G, and T nucleotides. **Genomic Foundation Models (GFMs)**, like **OmniGenome** (Yang et al., 2025), are large-scale models pre-trained on massive amounts of genomic sequences.

By learning the "grammar" of DNA, these models develop a powerful, generalized understanding of genomics. This makes them incredibly effective for *in-silico* (computational) sequence analysis. As noted in studies, GFMs can be fine-tuned to achieve state-of-the-art performance on various downstream tasks, including TFB prediction, without needing to be trained from scratch.

### 4. The Workflow: A 4-Step Guide to Fine-Tuning

In this tutorial, we will follow a standard 4-step fine-tuning pipeline, which applies to most of fine-tuning-based genomic tasks. This structured approach is a common practice in machine learning and will serve as our roadmap.

```mermaid
flowchart TD
    subgraph "4-Step Workflow for TE Prediction"
        A["📥 Step 1: Data Preparation<br/>Download and process the TE dataset"] --> B["🔧 Step 2: Model Initialization<br/>Load the pre-trained OmniGenome model"]
        B --> C["🎓 Step 3: Model Training<br/>Fine-tune the model on the TE dataset"]
        C --> D["🔮 Step 4: Model Inference<br/>Use the trained model to predict TE on new sequences"]
    end

    style A fill:#e1f5fe,stroke:#333,stroke-width:2px
    style B fill:#f3e5f5,stroke:#333,stroke-width:2px
    style C fill:#e8f5e8,stroke:#333,stroke-width:2px
    style D fill:#fff3e0,stroke:#333,stroke-width:2px
```

In a nutshell, we will see the following steps:
1. **[Data Preparation](./01_data_preparation.ipynb)**: Download and preprocess the DeepSEA dataset.
2. **[Model Initialization](./02_model_initialization.ipynb)**: Load the pre-trained OmniGenome model and set it up for our specific task.
3. **[Training Implementation](./03_training_implementation.ipynb)**: Fine-tune the model using our dataset and validate its performance.
4. **[Inference: Make Predictions](./04_inference.ipynb)**: Use the trained model to predict TF binding sites on new DNA sequences.


Now, let's get started with setting up our environment!



## 🚀 Step 1: Data Preparation

This first step is all about getting our data ready for in-silico analysis. It involves four key parts:
1.  **Environment Setup**: Installing and importing the necessary libraries.
2.  **Configuration**: Defining all our important parameters in one place.
3.  **Data Acquisition**: Downloading and preparing the raw dataset.
4.  **Data Loading**: Creating a pipeline to efficiently feed data to the model.

### 1.1: Environment Setup

First, let's install the required Python packages. `omnigenbench` is our core library, `transformers` provides the underlying model architecture, and the other packages are utilities for our workflow.

In [ ]:
!pip install -U numpy transformers omnigenbench autocuda findfile requests


Next, we import the libraries we just installed. This gives us the tools for data processing, deep learning, and interacting with the operating system.

A key part of this setup is determining the best available hardware for training. Our script will automatically prioritize a **CUDA-enabled GPU** if one is available, as this can accelerate training by 10-100x compared to a CPU. This makes a huge difference when working with large models and datasets.

In [ ]:
import os
import json
import torch
import numpy as np
import requests
import zipfile
from typing import Optional, List, Dict, Any

from omnigenbench import (
    OmniDataset,
    ClassificationMetric,
    AccelerateTrainer,
    ModelHub,
    OmniModelForMultiLabelSequenceClassification,
    OmniTokenizer,
)

from transformers import AutoTokenizer, AutoModel
import autocuda
import findfile

DEVICE = autocuda.auto_cuda()

### 1.2: Global Configuration

To make our tutorial easy to modify and understand, we'll centralize all important parameters in this section. This is a best practice in software development that makes experiments more reproducible.

#### Key Parameters
-   **Dataset**: We define the local path and download URL for our dataset.
-   **Model**: We select which pre-trained OmniGenome model to use. For this tutorial, we'll start with `OmniGenome-52M` because it's fast and efficient, making it perfect for learning and prototyping.
-   **Training Hyperparameters**: These values, like `MAX_LENGTH`, `BATCH_SIZE`, and `LEARNING_RATE`, control the training process. They have been set to sensible defaults that work well for this task.
-   **Data Sampling**: For a quick test run, we'll only use `1000` samples and `10` TF labels. To run a full training session, you can comment out these lines to use the entire dataset.

This centralized approach allows you to easily experiment with different settings (e.g., a larger model, a different learning rate) without having to hunt through the code.

#### Note
Almost all the parameters here are standard in machine learning workflows and have a default value that works well if you don't set them. Don't worry if some of these terms are unfamiliar right now. We'll explain each one as we go along.

In [ ]:
# Configuration for Complete TFB Tutorial
class CompleteTFBConfig:
    # Dataset
    DATASET_NAME = "yangheng/deepsea_tfb_prediction"
    LOCAL_CACHE_DIR = "deepsea_tfb_prediction"
    
    # Model
    MODEL_NAME = "yangheng/OmniGenome-52M"
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    LEARNING_RATE = 5e-5
    WEIGHT_DECAY = 1e-3
    EPOCHS = 10
    PATIENCE = 3
    
    # Quick testing configuration
    NUM_TF_LABELS = 10  # Use 10 TFs for quick testing
    MAX_EXAMPLES = 1000  # Use 1000 samples for quick testing
    
    # Output
    SAVE_MODEL_DIR = "tfb_trained_model"

config = CompleteTFBConfig()

print("⚙️ Complete TFB Tutorial Configuration:")
print(f"  📊 Dataset: {config.DATASET_NAME}")
print(f"  🧬 Model: {config.MODEL_NAME}")
print(f"  📏 Sequence length: {config.MAX_LENGTH}")
print(f"  🎯 TF labels: {config.NUM_TF_LABELS}")
print(f"  🔢 Sample limit: {config.MAX_EXAMPLES}")
print(f"  🏋️ Training epochs: {config.EPOCHS}")
print("✅ Configuration ready!")

### 1.3: Data Acquisition

With our environment configured, it's time to download the DeepSEA dataset. The function below automates this process by:
1.  Checking if the data already exists.
2.  Downloading the dataset from the specified URL if needed.
3.  Extracting the files.
4.  Cleaning up the temporary zip file.

This ensures we have the `train.jsonl`, `valid.jsonl`, and `test.jsonl` files ready for the next stage. These files represent the standard splits for training, validating, and testing our model.

In [ ]:
# Load tokenizer and datasets using enhanced OmniDataset
print("🔄 Loading tokenizer...")
tokenizer = OmniTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)
print("✅ Tokenizer loaded!")

print("? Loading DeepSEA TFB dataset with automatic download...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name=config.DATASET_NAME,
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH,
    cache_dir=config.LOCAL_CACHE_DIR
)

# Create DataLoaders
train_loader = datasets['train'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=True
)
valid_loader = datasets['valid'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=False
)
test_loader = datasets['test'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=False
)

print(f"? Dataset and DataLoaders created:")
print(f"  ? Training samples: {len(datasets['train'])}")
print(f"  🔍 Validation samples: {len(datasets['valid'])}")  
print(f"  🧪 Test samples: {len(datasets['test'])}")
print(f"  📦 Batch size: {config.BATCH_SIZE}")

# Show data sample
sample_batch = next(iter(train_loader))
print(f"\n? Sample batch structure:")
print(f"  ? Input shape: {sample_batch['input_ids'].shape}")
print(f"  🎯 Label shape: {sample_batch['labels'].shape}")
print("✅ Data preparation complete!")

### 1.4: Custom Dataset and Data Loaders

Now that we have the data files, we need a way to load them into our model efficiently. We'll do this in two parts:

#### A. The `DeepSEADataset` Class
This custom class acts as a bridge between our raw `.jsonl` files and the PyTorch ecosystem. It inherits from `OmniDataset` and tells our framework how to process each data entry. Specifically, it:
1.  **Processes DNA sequence and its labels** via `prepare_input()`.
2.  **Truncates or pads the sequence** to a fixed length (`MAX_LENGTH`). This is crucial because language models require inputs of a consistent size. We use center cropping, assuming the most important information is in the middle of the sequence.
3.  **Selects the specific TF labels** we want to train on.
4.  **Tokenizes the sequence**, converting the string of "A, C, G, T" into numerical tokens that the model can understand.
5.  **Formats the output** as PyTorch tensors.

#### B. Creating DataLoaders
Once the `DeepSEADataset` is defined, we use PyTorch's `DataLoader` to create an efficient pipeline. The `DataLoader` is responsible for:
1.  **Batching**: Grouping individual samples into batches (`BATCH_SIZE`). This is far more efficient for GPU processing than handling samples one by one.
2.  **Shuffling**: Randomly shuffling the training data each epoch to prevent the model from learning the order of the data, which helps improve generalization.
3.  **Parallelism**: Loading data in the background so it's ready for the model when needed (though we use a single worker here to ensure compatibility).

These components create a robust and efficient data pipeline, which is a cornerstone of any deep learning project.

In [ ]:
class DeepSEADataset(OmniDataset):
    def __init__(
        self,
        data_source: str,
        tokenizer,
        max_length: Optional[int] = None,
        label_indices: Optional[List[int]] = None,
        **kwargs
    ):
        super().__init__(data_source, tokenizer, max_length, **kwargs)
        self.label_indices = label_indices

        for key, value in kwargs.items():
            self.metadata[key] = value

    def prepare_input(self, instance: Dict[str, Any], **kwargs) -> Dict[str, torch.Tensor]:
        """
        This is the only method you need to customize for your dataset. 
        It takes a single data instance (a dictionary) and processes 
        it to return a dictionary of torch tensors ready for model input.
        Imagine you have a dataset where each instance is a dictionary like:
        {
            "sequence": "ACGTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGC...",
            "label": [0, 1, 0, 0, 1, ..., 0],  # Multi-hot encoded labels for TF binding
            "auxiliary_info": "some additional info",  # e.g., matrix or file path to some extra data
        },
        You can access the sequence and labels using instance['sequence'] and instance['label'].
        """
        def truncate_sequence(seq: str, max_len: Optional[int]) -> str:
            if max_len is None:
                return seq
            if len(seq) == max_len:
                return seq
            elif len(seq) > max_len:
                start_idx = (len(seq) - max_len) // 2
                return seq[start_idx:start_idx + max_len]
            else:
                return seq + ("N" * (max_len - len(seq)))

        sequence = instance.get('sequence') or instance.get('seq')
        labels = instance.get('label', None) if 'label' in instance else instance.get('labels', None)

        tokenized_inputs = self.tokenizer(
            truncate_sequence(sequence, self.max_length),
            padding="do_not_pad",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            **kwargs
        )

        labels_tensor = None
        if labels is not None:
            labels_tensor = torch.tensor(labels, dtype=torch.float32)
            if hasattr(self, 'label_indices') and self.label_indices is not None:
                labels_tensor = labels_tensor[torch.tensor(self.label_indices, dtype=torch.long)]

        tokenized_inputs["labels"] = labels_tensor

        for key in tokenized_inputs:
            if isinstance(tokenized_inputs[key], torch.Tensor) and tokenized_inputs[key].ndim > 1:
                tokenized_inputs[key] = tokenized_inputs[key].squeeze(0)

        return tokenized_inputs

print("📝 Custom dataset class definition completed!")

# Now, let's create the datasets and data loaders
train_loader, valid_loader, test_loader, dataset_info = create_datasets_and_loaders(
    tokenizer=tokenizer, train_file=train_file, valid_file=valid_file, test_file=test_file,
    max_length=MAX_LENGTH, max_examples=MAX_EXAMPLES, label_indices=LABEL_INDICES, batch_size=BATCH_SIZE
)

print("\n🎉 Step 1 complete: Data Preparation is finished!")
print(f"📈 Training samples: {dataset_info['train_size']}")
print(f"🔍 Validation samples: {dataset_info['valid_size']}")
print(f"🧪 Test samples: {dataset_info['test_size']}")

## 🚀 Step 2: Model Initialization

With our data pipeline in place, it's time to set up the model. This is where the power of Genomic Foundation Models (GFMs) comes into play. Instead of building a model from scratch, we will load the pre-trained **OmniGenome** model and adapt it for our specific task.

This process involves three key components:
1.  **The Tokenizer**: This is the same tokenizer we used in the data preparation step. It's responsible for converting DNA sequences into a numerical format the model can process. It's crucial that we use the *exact same* tokenizer that the model was pre-trained with.
2.  **The Base Model**: This is the core OmniGenome model, which has already learned the fundamental patterns of genomic sequences from its extensive pre-training. We load it directly from the Hugging Face Hub.
3.  **The Classification Head**: The base model is great at understanding sequences, but it doesn't know anything about our specific TFB prediction task. To fix this, we add a "classification head" on top of the base model. This is a simple neural network layer that takes the sequence representation from the base model and maps it to our desired output, in this case, a prediction for each of the TF labels.


The `OmniModelForMultiLabelSequenceClassification` class from our library handles this process for us, seamlessly combining the base model and the new classification head.
In adddition, most of the models and datasets have been integrated into the OmniGenBench package, making it easy to load them with just a few lines of code. Please refer to the sub-tutorials of [data preparation](./02_data_preparation.ipynb) and [model initialization](./03_model_initialization.ipynb) for more details.

In [ ]:
# === Model Initialization ===
# We almost support all genomic foundation models from Hugging Face Hub.
print(f"🔧 Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"🧠 Loading base model: {MODEL_NAME}")
base_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"🎯 Creating {num_labels}-label classification head")
model = OmniModelForMultiLabelSequenceClassification(
    base_model,
    tokenizer,
    num_labels=len(LABEL_INDICES), 
)

if device is not None:
    model = model.to(device).to(torch.float32)

print(f"✅ Model configuration completed!")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model statistics:")
print(f"  🔢 Total parameters: {total_params:,}")
print(f"  🎓 Trainable parameters: {trainable_params:,}")
print(f"  💾 Model size: ~{total_params * 4 / 1e9:.1f}GB")


## 🚀 Step 3: Model Training

This is the most exciting part! With our data and model ready, we can now begin the **fine-tuning** process. During training, the model will learn to associate specific patterns in the DNA sequences with the presence or absence of TF binding sites.

### Our Training Strategy

We won't just train the model blindly. We'll use a sophisticated strategy to ensure the best possible outcome:

1.  **Optimizer (AdamW)**: We use the AdamW optimizer, a popular choice for training transformer models. It intelligently adapts the learning rate for different parameters and includes a technique called "weight decay" to help prevent overfitting.
2.  **Evaluation Metric (ROC-AUC)**: For a multi-label classification task like this, accuracy can be misleading, especially if the data is imbalanced (e.g., more non-binding sites than binding sites). The **ROC-AUC score** is a much better metric. It measures the model's ability to distinguish between positive and negative classes, regardless of their balance. A score of 1.0 is perfect, while 0.5 is equivalent to random guessing. **OmniGenBench supports over 60+ ML metrics and custmized metrics for different tasks.**
3.  **Early Stopping**: Training for too long can cause the model to "memorize" the training data and perform poorly on new, unseen data (a problem known as overfitting). To prevent this, we use **early stopping**. We monitor the ROC-AUC score on our validation set after each epoch. If the score doesn't improve for a set number of epochs (our `patience` parameter), we stop the training and save the best-performing version of the model.

The `AccelerateTrainer` from `omnigenbench` wraps all this logic into a simple interface, allowing us to launch the training process with just a few lines of code.

In [ ]:
metric_functions = [ClassificationMetric(ignore_y=-100).roc_auc_score]
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

if os.path.exists(SAVE_MODEL_DIR):
    print(f"📁 Found existing model: {SAVE_MODEL_DIR}")
    print("🔄 Skipping training (delete directory to retrain)")
    training_results = {"status": "skipped", "reason": "model_exists", "save_dir": SAVE_MODEL_DIR}
else:
    trainer = AccelerateTrainer(
        model=model, train_loader=train_loader, eval_loader=valid_loader, test_loader=test_loader,
        optimizer=optimizer, epochs=EPOCHS, compute_metrics=metric_functions,
        patience=PATIENCE, device=DEVICE, seed=42
    )

    print("🎓 Starting training...")

    metrics = trainer.train()

    print("💾 Saving trained model...")
    trainer.save_model(SAVE_MODEL_DIR)

    print(f"✅ Training completed successfully!")
    print(f"📂 Model saved to: {SAVE_MODEL_DIR}")

    training_results = {"status": "completed", "metrics": metrics, "save_dir": SAVE_MODEL_DIR}

## 🔮 Step 4: Model Inference and Interpretation

Now that we have a trained model, let's use it for its intended purpose: predicting TF binding sites on new DNA sequences. This process is called **inference**.

### The Inference Pipeline

Our inference pipeline consists of a few key steps:
1.  **Load the Model**: We load the best-performing model that was saved during training.
2.  **Process the Input**: We take a new DNA sequence and apply the same preprocessing steps we used for our training data (truncating/padding and tokenizing).
3.  **Run Prediction**: We feed the processed sequence to the model and get its predictions. We use `torch.no_grad()` to disable gradient calculations, which makes inference faster and uses less memory.
4.  **Interpret the Results**: The model's raw output is a set of probabilities. We'll interpret these to make them more understandable, identifying which TFs are predicted to bind and with what level of confidence.

To demonstrate, we'll test our model on a few sample sequences and print out a user-friendly summary of the results. This shows how the model can be used in a real-world application to analyze sequences of interest.

In [ ]:

print("🔄 Falling back to pre-trained model...")
inference_model = ModelHub.load("yangheng/ogb_tfb_finetuned")
inference_model = inference_model.to(DEVICE)
inference_model.eval()
print("✅ Fallback model loaded successfully")

sample_sequences = {
    "Random sequence": "AGCT" * (128 // 4),
    "AT-rich sequence": "AATT" * (128 // 4),
    "GC-rich sequence": "GCGC" * (128 // 4),
}

for seq_name, sequence in sample_sequences.items():
    print(f"\n{'='*50}")
    print(f"🔬 Test sequence: {seq_name}")
    print(f"📏 Sequence length: {len(sequence)} bp")
    print(f"📝 First 20 bases: {sequence[:20]}...")

    # —— 预处理（中心截断 / N 填充）——
    if len(sequence) > MAX_LENGTH:
        start = (len(sequence) - MAX_LENGTH) // 2
        processed_seq = sequence[start:start + MAX_LENGTH]
    elif len(sequence) < MAX_LENGTH:
        processed_seq = sequence + "N" * (MAX_LENGTH - len(sequence))
    else:
        processed_seq = sequence

    print(f"🧬 Analyzing sequence of length {len(sequence)} bp")

    print("🔮 Running prediction...")
    with torch.no_grad():
        outputs = inference_model.inference(sequence)
    print("✅ Prediction completed!")

    # —— 结果解释 —— 
    predictions = outputs.get('predictions', None)
    probabilities = outputs.get('probabilities', None)

    if predictions is None:
        print("❌ No data found in prediction results")
        continue

    predictions = np.array(predictions)
    probabilities = np.array(probabilities) if probabilities is not None else None

    positive_count = np.sum(predictions == 1)
    total_count = len(predictions)

    print(f"📊 Prediction summary:")
    print(f"  🎯 Total TF binding sites analyzed: {total_count}")
    print(f"  ✅ Predicted binding sites: {positive_count}")
    print(f"  📈 Binding rate: {positive_count/total_count:.1%}")

    if probabilities is not None:
        print(f"\n🏆 Top 5 highest confidence predictions:")
        sorted_indices = np.argsort(probabilities)[::-1]

        for i, idx in enumerate(sorted_indices[:5]):
            tf_id = idx + 1
            prediction = "Binding" if predictions[idx] == 1 else "No binding"
            confidence = probabilities[idx]

            if confidence > 0.8:
                emoji = "🔥"
            elif confidence > 0.6:
                emoji = "⭐"
            elif confidence > 0.4:
                emoji = "💫"
            else:
                emoji = "💭"

            print(f"  {emoji} TF-{tf_id:03d}: {prediction} (confidence: {confidence:.3f})")


## 🎉 Tutorial Summary and Next Steps

Congratulations! You have successfully completed this educational tutorial on transcription factor binding prediction with OmniGenBench.

### What You've Learned

You've walked through a complete, end-to-end MLOps workflow, a critical skill in modern computational biology. Specifically, you have:
1.  **Understood the "Why"**: Gained an appreciation for the biological problem of TFB prediction and how Genomic Foundation Models provide a powerful solution.
2.  **Mastered the 4-Step Workflow**:
    -   **Step 1: Data Preparation**: You learned how to acquire, process, and efficiently load a large-scale genomic dataset.
    -   **Step 2: Model Initialization**: You saw how to leverage a powerful pre-trained model and adapt it for a specific downstream task.
    -   **Step 3: Model Training**: You implemented a robust training strategy with best practices like early stopping and appropriate evaluation metrics.
    -   **Step 4: Model Inference**: You used your fine-tuned model to make predictions on new data and interpreted the results.

### Where to Go from Here?

This tutorial is just the beginning. Here are some ideas for your next steps:

-   **Scale Up**: Go back to the configuration cell, comment out the `MAX_EXAMPLES` and `NUM_TF_LABELS` lines, and train a full model on the entire dataset.
-   **Experiment with Different Models**: Try using a larger, more powerful GFM like `OmniGenome-186M` to see if it improves performance.
-   **Tune Hyperparameters**: Adjust parameters like the `LEARNING_RATE` or `BATCH_SIZE` to see how they affect the training outcome.
-   **Analyze Your Own Data**: Adapt the data loading pipeline to work with your own DNA sequences of interest.
-   **Explore Other Tasks**: OmniGenBench can be applied to many other genomic tasks. Check out the documentation for more examples and tutorials.

Thank you for following along. We hope this tutorial has provided you with the knowledge and confidence to apply deep learning to your own genomics research. Happy coding!